## Bonus: Evaluation and Production Patterns

This notebook is a reference guide — you don't need to run it live. It covers the production failure modes that bite every team building agentic RAG systems, and the retrieval evaluation metrics you need to know whether your system is actually working.

**Key concepts**: intent bleed, slot hallucination, retrieval evaluation, MRR, NDCG, precision@k

## Retrieval Evaluation Metrics

If you can't measure it, you can't improve it. These are the metrics that matter for retrieval quality.

In [ ]:
def reciprocal_rank(results: list[str], relevant: set[str]) -> float:
    """MRR: How high up is the first relevant result? Higher is better. Max = 1.0"""
    for i, r in enumerate(results, 1):
        if r in relevant:
            return 1.0 / i
    return 0.0


def precision_at_k(results: list[str], relevant: set[str], k: int) -> float:
    """Precision@k: What fraction of the top k results are relevant?"""
    top_k = results[:k]
    return sum(1 for r in top_k if r in relevant) / k


def ndcg_at_k(results: list[str], relevant: set[str], k: int) -> float:
    """NDCG@k: Are the best results ranked at the top? Higher is better. Max = 1.0"""
    import math
    dcg = sum(
        (1 if results[i] in relevant else 0) / math.log2(i + 2)
        for i in range(min(k, len(results)))
    )
    ideal = sum(1 / math.log2(i + 2) for i in range(min(len(relevant), k)))
    return dcg / ideal if ideal > 0 else 0.0


# Example: searching for "black dress" returned these products
results = ["Washington dress", "Colette dress", "GLASSIG ESPADRILLE", "Rihanna dress", "Atlas jacket"]
relevant = {"Washington dress", "Colette dress", "Rihanna dress"}  # ground truth

print(f"MRR:           {reciprocal_rank(results, relevant):.3f}  (first relevant result at rank 1 = perfect)")
print(f"Precision@3:   {precision_at_k(results, relevant, k=3):.3f}  (2 of top 3 are relevant)")
print(f"Precision@5:   {precision_at_k(results, relevant, k=5):.3f}  (3 of top 5 are relevant)")
print(f"NDCG@5:        {ndcg_at_k(results, relevant, k=5):.3f}  (relevant results near top = high score)")

## Production Failure Modes

### Intent Bleed
When context from a previous intent leaks into the current one. The slot filler starts suggesting values from a billing conversation during a product search.

### Slot Hallucination
When the LLM fills in slot values that the user never provided. "I see you're looking for contracts from March 2023" — when the user never mentioned a date.

### Low-Confidence Deadlocks
When the intent classifier can't decide between two intents and the system keeps asking the user to clarify without making progress.

### Over-Eager Reclassification
When the intent classifier reclassifies every follow-up message as a new intent instead of treating it as continuation of the current conversation.

In [ ]:
# ── Failure Mode 1: Intent Bleed ─────────────────────────────────────────────
# Slots from a previous intent leak into the next one.
# Fix: always call session.reset_intent() on intent switch — which our orchestrator does.

print("=== Intent Bleed ===")
print("User: 'I want to return my jacket'  → intent: support, slots: {issue: 'return', product: 'jacket'}")
print("User: 'Show me blue trousers'        → intent switches to product")
print("WITHOUT reset: slots still contain {issue: 'return'} — support context bleeds into product search")
print("WITH reset:    slots cleared on switch — clean product search")
print()

# ── Failure Mode 2: Slot Hallucination ───────────────────────────────────────
# The LLM fills in slot values the user never provided.
# Fix: strict instructions — "NEVER hallucinate slot values, only extract what the user said"

print("=== Slot Hallucination ===")
print("User: 'I'm looking for something nice'")
print("BAD:  extracted_slots = {'product_keyword': 'dress', 'colour': 'white'}  ← hallucinated")
print("GOOD: extracted_slots = {}  missing_required = ['product_keyword']       ← asks for clarification")
print()

# ── Failure Mode 3: Low-Confidence Deadlock ──────────────────────────────────
# Classifier can't decide between intents and keeps asking the user to clarify.
# Fix: CONFIDENCE_THRESHOLD + fallback response that names the options clearly.

print("=== Low-Confidence Deadlock ===")
print("User: 'Can you help me?'  → confidence: 0.3 (below threshold of 0.6)")
print("GOOD: 'I can help with product search, billing questions, or order issues. What do you need?'")
print("BAD:  keep re-asking 'What do you mean?' every turn indefinitely")
print()

# ── Failure Mode 4: Over-Eager Reclassification ──────────────────────────────
# Every follow-up message gets classified as a new intent instead of continuing.
# Fix: INTENT_SWITCH_THRESHOLD (0.7) — only switch if new intent is very confident.

print("=== Over-Eager Reclassification ===")
print("User: 'Show me black jackets'     → intent: product (0.9)")
print("User: 'What about blue ones?'     → classifier might say product (0.6) or unknown (0.5)")
print("WITHOUT threshold: resets slots — loses 'jacket' context")
print("WITH threshold:    stays in product intent, adds colour='blue' to existing slots")

## Key Takeaways

- **MRR (Mean Reciprocal Rank)** — How high up is the first relevant result?
- **NDCG (Normalized Discounted Cumulative Gain)** — Are the best results at the top?
- **Precision@k** — What fraction of the top k results are relevant?

These metrics tell you whether your retrieval pipeline is actually working — before the LLM ever sees the context.

---

## Opik: LLM-Based RAG Evaluation

The metrics above tell you about retrieval ranking. Opik goes a step further as it uses an LLM to evaluate the *quality* of your RAG responses: did the answer use the retrieved context, is it relevant to the question, and did the model hallucinate anything?

Additionally, it creates traces for everything! This means real-time monitoring of your entire system end-to-end. You can know how much each call costs, you can see what questions are being asked by your users, and you can see timings on latency. Everything is tracked for you with traces, including individual spans. 

Additionally, it allows for end-to-end evaluations of your system by running a simple script. This allows you to see when changing retrieval breaks something downstream, or if adding a reranker actually improves your retrieval and subsequent generated responses. 

**Before running these cells:** make sure you have `OPIK_API_KEY` set in your `.env` file.  
Create a free account at [https://www.comet.com/signup](https://www.comet.com/signup) and get your key from [Account Settings](https://www.comet.com/account-settings).

In [ ]:
import os
import opik
from dotenv import load_dotenv

load_dotenv()

# Configure Opik with your API key from .env
os.environ["OPIK_API_KEY"] = os.getenv("OPIK_API_KEY", "")

if not os.environ["OPIK_API_KEY"]:
    print("No OPIK_API_KEY found. Add it to your .env file.")
else:
    print("Opik configured. Results will be logged to your Comet dashboard.")

In [ ]:
# Create an Opik dataset with RAG examples to evaluate.
# Each item has: input (the user question), context (what was retrieved),
# and expected_output (the ideal answer we'd expect).

client = opik.Opik()

dataset_items = [
    {
        "input": "Show me black dresses for a cocktail party",
        "context": [
            "Washington dress (Dress, Black): Short-sleeved dress in stretch fabric with a fitted silhouette, suitable for formal occasions.",
            "Colette dress (Dress, Black): Elegant sleeveless dress in woven fabric with a V-neckline and a flared skirt.",
        ],
        "expected_output": "Here are some black dresses suitable for a cocktail party: the Washington dress is a fitted short-sleeved option, and the Colette dress is an elegant sleeveless style with a V-neckline."
    },
    {
        "input": "I need warm layering pieces for a ski trip, nothing too dark",
        "context": [
            "Greyson fleece (Sweater, Light Grey): Soft fleece mid-layer with zip neck, ideal for layering in cold conditions.",
            "Nordic cardigan (Cardigan, Beige): Chunky knit cardigan in a neutral tone, warm and versatile.",
        ],
        "expected_output": "For ski trip layering in lighter tones, the Greyson fleece in light grey is a great mid-layer, and the Nordic cardigan in beige is a warm, versatile option."
    },
    {
        "input": "Find me smart casual men's trousers",
        "context": [
            "Chino slim (Trousers, Dark Beige): Slim-fit chinos in stretch cotton, versatile for smart casual wear.",
            "York trousers (Trousers, Grey): Tailored trousers in a wool blend, suitable for office or smart casual settings.",
        ],
        "expected_output": "For smart casual men's trousers, the Chino slim in dark beige offers a versatile slim fit, while the York trousers in grey provide a more tailored look."
    },
]

# Push dataset to Opik
dataset = client.get_or_create_dataset(name="rag-evaluation-workshop")
dataset.insert(dataset_items)
print(f"Dataset '{dataset.name}' ready with {len(dataset_items)} items.")

In [ ]:
from opik.evaluation.metrics import ContextPrecision, AnswerRelevance, Hallucination
from opik.evaluation import evaluate

# Simulate the RAG pipeline — in production this would call your actual agent
def rag_task(item: dict) -> dict:
    """Takes a dataset item and returns the model's response.
    Here we return the expected_output to simulate a well-behaved system.
    In production, replace this with your actual agent call."""
    return {
        "input": item["input"],
        "output": item["expected_output"],
        "context": item["context"],
    }

# Run evaluation with all three metrics
results = evaluate(
    dataset=dataset,
    task=rag_task,
    scoring_metrics=[
        ContextPrecision(),   # Did the retrieved context actually support the answer?
        AnswerRelevance(),    # Is the answer relevant to the original question?
        Hallucination(),      # Did the model make up anything not in the context?
    ],
    experiment_name="workshop-baseline",
)

print("Evaluation complete. View full results in your Comet dashboard.")

### What to look for in your Opik dashboard

- **Context Precision** — Are the retrieved chunks actually relevant to the query? Low scores mean your filters or embeddings are pulling in noise.
- **Answer Relevance** — Is the final LLM response on-topic? Low scores often mean your system prompt needs tightening.
- **Hallucination** — Is the model inventing facts not present in the retrieved context? Any score above 0 here is a production risk.

These three metrics together give you a full picture of where your RAG pipeline is breaking down — retrieval, generation, or both.